In [1]:
#Products Data — Complete Cleaning Pipeline

import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load
products = pd.read_csv('KartZone_Products_V2.csv')

# First look
print(products.shape)
print(products.head())
print(products.info())
print(products.describe())
print(products.columns.tolist())

(815, 14)
  Product_ID                 Product_Name     Category     Sub_Category  \
0  ELEC-1000             Sony WiFi Router  electronics      WiFi Router   
1  ELEC-1001            OnePlus Pen Drive  Electronics        Pen Drive   
2  FASH-2000       Peter England Leggings      fashion         Leggings   
3  ELEC-1002  Asus Tablet 32GB WiFi Black  Electronics           Tablet   
4  HOME-3000     Prestige Pressure Cooker         Home  Pressure Cooker   

           Brand          MRP Cost_Price Discount_Pct Selling_Price  \
0           Sony     ₹5493.59    3620.92           15       4669.55   
1        OnePlus    Rs.909.98        NaN           5%        864.48   
2  Peter England       627.31        NaN          10%       ₹564.58   
3           Asus     61954.36   39675.49          10%      55758.92   
4       Prestige  INR 2715.15    1906.06          NaN      ₹2307.88   

  Launch_Date  Rating  Stock_Units Weight_KG Product_Status  
0  02/01/2022     4.4         78.0       0.4      

In [2]:
#Understand Data Quality

# Missing values
print(products.isnull().sum())
print(products.isnull().sum() / len(products) * 100)

# Duplicates
print(f"Duplicate rows: {products.duplicated().sum()}")

# Blank rows
print(f"Blank rows: {products.isnull().all(axis=1).sum()}")

# Unique values per column
print(products.nunique())

# Sample messy values per column
for col in products.columns:
    print(f"\n{col}:")
    print(products[col].value_counts(dropna=False).head(8))

Product_ID          0
Product_Name        0
Category            0
Sub_Category        0
Brand               0
MRP                44
Cost_Price         68
Discount_Pct       68
Selling_Price      87
Launch_Date       177
Rating             85
Stock_Units        94
Weight_KG          88
Product_Status     90
dtype: int64
Product_ID         0.000000
Product_Name       0.000000
Category           0.000000
Sub_Category       0.000000
Brand              0.000000
MRP                5.398773
Cost_Price         8.343558
Discount_Pct       8.343558
Selling_Price     10.674847
Launch_Date       21.717791
Rating            10.429448
Stock_Units       11.533742
Weight_KG         10.797546
Product_Status    11.042945
dtype: float64
Duplicate rows: 15
Blank rows: 0
Product_ID        800
Product_Name      565
Category           20
Sub_Category       60
Brand              60
MRP               757
Cost_Price        732
Discount_Pct       29
Selling_Price     713
Launch_Date       594
Rating             

In [3]:
# Remove Duplicates and Blank Rows
print(f"Before: {len(products)}")

# Remove blank rows first
products = products.dropna(how='all')

# Remove exact duplicates
products = products.drop_duplicates()

# Remove duplicate Product_IDs — keep first
products = products.drop_duplicates(subset=['Product_ID'], keep='first')

# Reset index
products = products.reset_index(drop=True)

print(f"After: {len(products)}")

Before: 815
After: 800


In [4]:
#Clean Product_ID

# Check format
print(products['Product_ID'].value_counts().head(10))

# Strip whitespace
products['Product_ID'] = products['Product_ID'].astype(str).str.strip()

# Check for nulls or INVALID entries
invalid_ids = products['Product_ID'].str.contains('INVALID', na=False)
print(f"Invalid Product IDs: {invalid_ids.sum()}")

# Remove invalid product rows
products = products[~invalid_ids].reset_index(drop=True)

print(f"Products after removing invalid IDs: {len(products)}")

Product_ID
ELEC-1000    1
ELEC-1180    1
HOME-3089    1
ELEC-1177    1
BEAU-4090    1
ELEC-1178    1
HOME-3090    1
FASH-2171    1
BEAU-4091    1
HOME-3091    1
Name: count, dtype: int64
Invalid Product IDs: 0
Products after removing invalid IDs: 800


In [5]:
# Clean Product_Name

# Strip whitespace and fix case
products['Product_Name'] = (
    products['Product_Name']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# Replace missing encodings
products['Product_Name'] = products['Product_Name'].replace(
    ['nan','NaN','NA','N/A','None','','null','-'], np.nan
)

# Fill missing with Category + Sub_Category
products['Product_Name'] = products.apply(
    lambda row: f"{row['Brand']} {row['Sub_Category']}"
    if pd.isnull(row['Product_Name']) else row['Product_Name'],
    axis=1
)

print(products['Product_Name'].head(10))

0               Sony WiFi Router
1              OnePlus Pen Drive
2         Peter England Leggings
3    Asus Tablet 32GB WiFi Black
4       Prestige Pressure Cooker
5                Nike Sunglasses
6                  Arrow Sandals
7                  Asus Smart TV
8                    Sony Camera
9            Story@Home LED Lamp
Name: Product_Name, dtype: object


In [6]:
#Standardize Category

def standardize_category(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if 'electron' in val:                    return 'Electronics'
    if 'fashion' in val or 'cloth' in val:   return 'Fashion'
    if 'home' in val or 'kitchen' in val:    return 'Home & Kitchen'
    if 'beauty' in val or 'personal' in val: return 'Beauty'
    return np.nan

products['Category'] = products['Category'].apply(standardize_category)

# Fill nulls — derive from Product_ID prefix
def derive_cat_from_id(pid):
    if pd.isnull(pid): return np.nan
    pid = str(pid).upper()
    if pid.startswith('ELEC'): return 'Electronics'
    if pid.startswith('FASH'): return 'Fashion'
    if pid.startswith('HOME'): return 'Home & Kitchen'
    if pid.startswith('BEAU'): return 'Beauty'
    return np.nan

products['Category'] = products.apply(
    lambda row: derive_cat_from_id(row['Product_ID'])
    if pd.isnull(row['Category']) else row['Category'],
    axis=1
)

print(products['Category'].value_counts())

Category
Electronics       268
Fashion           253
Home & Kitchen    143
Beauty            136
Name: count, dtype: int64


In [7]:
# Clean MRP
def clean_price(val):
    if pd.isnull(val):
        return np.nan

    val = str(val).strip()

    # Remove currency labels and symbols, but keep the decimal point
    val = re.sub(r'₹|Rs\.?|INR|,', '', val, flags=re.IGNORECASE).strip()

    try:
        result = float(val)
        return result if result > 0 else np.nan
    except:
        return np.nan


products['MRP'] = products['MRP'].apply(clean_price)

# Check distribution
print(products['MRP'].describe())
print(f"Null MRP: {products['MRP'].isnull().sum()}")

# Fill nulls with category median
products['MRP'] = products.groupby('Category')['MRP'].transform(
    lambda x: x.fillna(x.median())
)

count       757.000000
mean       9788.138666
std       17495.416492
min         188.220000
25%        1405.980000
50%        3507.060000
75%        8609.900000
max      104842.740000
Name: MRP, dtype: float64
Null MRP: 43


In [8]:
# Clean Cost_Price
products['Cost_Price'] = products['Cost_Price'].apply(clean_price)

# Check for invalid negative cost prices
negative_cost = (products['Cost_Price'] < 0).sum()
print(f"Negative cost prices found: {negative_cost}")

# Cost cannot exceed MRP
invalid_cost = (products['Cost_Price'] > products['MRP']).sum()
print(f"Cost > MRP (before fix): {invalid_cost}")

# Fill nulls with category median
products['Cost_Price'] = products.groupby('Category')['Cost_Price'].transform(
    lambda x: x.fillna(x.median())
)

print(products[['MRP','Cost_Price']].describe())

Negative cost prices found: 0
Cost > MRP (before fix): 11
                 MRP    Cost_Price
count     800.000000    800.000000
mean     9533.207987   6217.102425
std     17062.368603  12309.326769
min       188.220000     83.750000
25%      1437.222500    832.947500
50%      3635.970000   2107.135000
75%      8376.810000   5131.407500
max    104842.740000  70579.880000


In [9]:
# Clean Discount_Pct and Selling_Price

# Clean Discount_Pct — remove % symbol
def clean_discount(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().replace('%','')
    try:
        d = float(val)
        # Valid range 0-70
        return d if 0 <= d <= 70 else np.nan
    except:
        return np.nan

products['Discount_Pct'] = products['Discount_Pct'].apply(clean_discount)

# Fill nulls with category median
products['Discount_Pct'] = products.groupby('Category')['Discount_Pct'].transform(
    lambda x: x.fillna(x.median())
)

# Recalculate Selling_Price from clean MRP and Discount
products['Selling_Price'] = products.apply(
    lambda row: round(row['MRP'] * (1 - row['Discount_Pct']/100), 2)
    if pd.notnull(row['MRP']) and pd.notnull(row['Discount_Pct']) else np.nan,
    axis=1
)

print(products[['MRP','Discount_Pct','Selling_Price']].head(10))

         MRP  Discount_Pct  Selling_Price
0   5493.590          15.0        4669.55
1    909.980           5.0         864.48
2    627.310          10.0         564.58
3  61954.360          10.0       55758.92
4   2715.150           5.0        2579.39
5   2607.470           5.0        2477.10
6   6407.840          10.0        5767.06
7   7977.265          15.0        6780.68
8  47545.800          15.0       40413.93
9   2714.640           5.0        2578.91


In [10]:
# Clean Rating

products['Rating'] = pd.to_numeric(products['Rating'], errors='coerce')

# Valid range 1.0 to 5.0
print(f"Rating > 5: {(products['Rating'] > 5).sum()}")
print(f"Rating < 1: {(products['Rating'] < 1).sum()}")

products['Rating'] = products['Rating'].where(
    (products['Rating'] >= 1.0) & (products['Rating'] <= 5.0),
    np.nan
)

# Fill nulls with category median
products['Rating'] = products.groupby('Category')['Rating'].transform(
    lambda x: x.fillna(round(x.median(),1))
)

print(products['Rating'].describe())

Rating > 5: 0
Rating < 1: 0
count    800.000000
mean       3.981250
std        0.626489
min        1.700000
25%        3.600000
50%        4.000000
75%        4.400000
max        5.000000
Name: Rating, dtype: float64


In [11]:
#Clean Stock_Units

products['Stock_Units'] = pd.to_numeric(
    products['Stock_Units'], errors='coerce'
)

# Negative stock — data error — replace with 0
print(f"Negative stock: {(products['Stock_Units'] < 0).sum()}")

# Fill nulls with 0
products['Stock_Units'] = products['Stock_Units'].fillna(0).astype(int)

print(products['Stock_Units'].describe())

Negative stock: 0
count    800.000000
mean     212.968750
std      160.576395
min        0.000000
25%       64.750000
50%      204.000000
75%      360.250000
max      499.000000
Name: Stock_Units, dtype: float64


In [12]:
#Clean Weight_KG

def clean_weight(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower().replace('kg','').strip()
    try:
        w = float(val)
        return w if w > 0 else np.nan
    except:
        return np.nan

products['Weight_KG'] = products['Weight_KG'].apply(clean_weight)

# Fill nulls with subcategory median
products['Weight_KG'] = products.groupby('Sub_Category')['Weight_KG'].transform(
    lambda x: x.fillna(x.median())
)

# If still null fill with category median
products['Weight_KG'] = products.groupby('Category')['Weight_KG'].transform(
    lambda x: x.fillna(x.median())
)

print(products['Weight_KG'].describe())

count    800.000000
mean       0.989244
std        2.232791
min        0.020000
25%        0.180000
50%        0.330000
75%        0.710000
max       17.220000
Name: Weight_KG, dtype: float64


In [13]:
#Parse Launch_Date

def parse_date(val):
    if pd.isnull(val) or str(val).strip() in ['','NA','N/A','nan','null','-']:
        return np.nan
    val = str(val).strip()
    formats = ['%Y-%m-%d','%d/%m/%Y','%d-%m-%Y',
               '%m/%d/%Y','%d.%m.%Y','%Y/%m/%d']
    for fmt in formats:
        try:
            return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
        except:
            continue
    return np.nan

products['Launch_Date'] = products['Launch_Date'].apply(parse_date)
products['Launch_Date'] = pd.to_datetime(products['Launch_Date'], errors='coerce')

# Fill missing with category median launch date
median_launch = products['Launch_Date'].dropna().sort_values()
mid_date = median_launch.iloc[len(median_launch)//2]
products['Launch_Date'] = products['Launch_Date'].fillna(mid_date)

print(products['Launch_Date'].describe())

count                    800
mean     2022-07-14 11:04:12
min      2020-01-08 00:00:00
25%      2021-09-19 12:00:00
50%      2022-07-09 00:00:00
75%      2023-05-26 18:00:00
max      2024-12-29 00:00:00
Name: Launch_Date, dtype: object


In [14]:
#Standardize Product_Status

def standardize_status(val):
    if pd.isnull(val):
        return 'Unknown'
    val = str(val).strip().lower()
    if 'active' in val and 'dis' not in val: return 'Active'
    if 'discontinue' in val:                 return 'Discontinued'
    if 'upcoming' in val:                    return 'Upcoming'
    if 'seasonal' in val:                    return 'Seasonal'
    if 'clearance' in val:                   return 'Clearance'
    if val in ['no','0','false']:            return 'Discontinued'
    return 'Active'   # default

products['Product_Status'] = products['Product_Status'].apply(
    standardize_status
)

print(products['Product_Status'].value_counts())

Product_Status
Active          351
Clearance        95
Seasonal         92
Discontinued     91
Unknown          89
Upcoming         82
Name: count, dtype: int64


In [15]:
# Outlier Validation

def check_outliers(series, name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((series < lower) | (series > upper)).sum()

    print(f"\n{name}:")
    print(f"  Bounds: {lower:.2f} to {upper:.2f}")
    print(f"  Potential outliers found: {outliers}")

print("\nChecking MRP:")
check_outliers(products['MRP'], 'MRP')

print("\nChecking Cost_Price:")
check_outliers(products['Cost_Price'], 'Cost_Price')

print("\nChecking Discount_Pct:")
check_outliers(products['Discount_Pct'], 'Discount_Pct')


Checking MRP:

MRP:
  Bounds: -8972.16 to 18786.19
  Potential outliers found: 83

Checking Cost_Price:

Cost_Price:
  Bounds: -5614.74 to 11579.10
  Potential outliers found: 85

Checking Discount_Pct:

Discount_Pct:
  Bounds: -17.50 to 42.50
  Potential outliers found: 12


In [16]:
# Feature Engineering

today = pd.Timestamp.now()

# 1. Profit Per Unit
products['Profit_Per_Unit'] = round(
    products['Selling_Price'] - products['Cost_Price'], 2
)

# 2. Gross Margin Percent
products['Gross_Margin_Pct'] = round(
    (products['Profit_Per_Unit'] / products['Selling_Price']) * 100, 2
)

# 3. Is Loss Maker
products['Is_Loss_Maker'] = (
    products['Selling_Price'] < products['Cost_Price']
).astype(int)

print(f"Loss-making products: {products['Is_Loss_Maker'].sum()}")

# 4. Product Age in Days
products['Product_Age_Days'] = (
    today - products['Launch_Date']
).dt.days

# 5. Product Age Band
def age_band(days):
    if pd.isnull(days): return 'Unknown'
    if days < 180:   return 'New (< 6 months)'
    elif days < 365: return 'Growing (6-12 months)'
    elif days < 730: return 'Mature (1-2 years)'
    else:            return 'Established (2+ years)'

products['Product_Age_Band'] = products['Product_Age_Days'].apply(age_band)

# 6. Rating Band
def rating_band(r):
    if pd.isnull(r): return 'Unknown'
    if r >= 4.5:   return 'Excellent'
    elif r >= 4.0: return 'Good'
    elif r >= 3.0: return 'Average'
    else:          return 'Poor'

products['Rating_Band'] = products['Rating'].apply(rating_band)

# 7. Stock Status
def stock_status(s):
    if pd.isnull(s): return 'Unknown'
    if s == 0:     return 'Out of Stock'
    elif s <= 20:  return 'Low Stock'
    elif s <= 100: return 'Adequate'
    else:          return 'High Stock'

products['Stock_Status'] = products['Stock_Units'].apply(stock_status)

# 8. High Discount Flag
products['High_Discount_Flag'] = (
    products['Discount_Pct'] > 20
).astype(int)

# 9. Price Tier
def price_tier(mrp):
    if pd.isnull(mrp): return 'Unknown'
    if mrp < 500:      return 'Budget'
    elif mrp < 2000:   return 'Mid-Range'
    elif mrp < 10000:  return 'Premium'
    else:              return 'Luxury'

products['Price_Tier'] = products['Selling_Price'].apply(price_tier)
# 10. Discount Impact Score
# How much revenue lost per unit due to discount
products['Discount_Amount'] = round(
    products['MRP'] - products['Selling_Price'], 2
)

print(products[['Product_ID','Profit_Per_Unit','Gross_Margin_Pct',
                'Is_Loss_Maker','Rating_Band','Stock_Status',
                'High_Discount_Flag','Price_Tier']].head(10))

Loss-making products: 87
  Product_ID  Profit_Per_Unit  Gross_Margin_Pct  Is_Loss_Maker Rating_Band  \
0  ELEC-1000          1048.63             22.46              0        Good   
1  ELEC-1001         -5035.43           -582.48              1     Average   
2  FASH-2000         -1736.95           -307.65              1   Excellent   
3  ELEC-1002         16083.43             28.84              0        Good   
4  HOME-3000           673.33             26.10              0        Good   
5  FASH-2001           938.69             37.89              0     Average   
6  FASH-2002          2884.04             50.01              0     Average   
7  ELEC-1003        -14460.29           -213.26              1        Good   
8  ELEC-1004          8996.74             22.26              0        Good   
9  HOME-3001          1179.33             45.73              0        Good   

  Stock_Status  High_Discount_Flag Price_Tier  
0     Adequate                   0    Premium  
1   High Stock      

In [17]:
#Normalization

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max — Rating and Discount_Pct
scaler_mm = MinMaxScaler()
products['Rating_Normalized']   = scaler_mm.fit_transform(products[['Rating']])
products['Discount_Normalized'] = scaler_mm.fit_transform(products[['Discount_Pct']])

# Z-Score — MRP and Profit_Per_Unit
scaler_z = StandardScaler()
products['MRP_Zscore']    = scaler_z.fit_transform(products[['MRP']])
products['Profit_Zscore'] = scaler_z.fit_transform(products[['Profit_Per_Unit']])

# Log Transform — MRP (right skewed)
products['MRP_Log']   = np.log1p(products['MRP'])
products['Cost_Log']  = np.log1p(products['Cost_Price'])
products['Stock_Log'] = np.log1p(products['Stock_Units'])

print(products[['Rating_Normalized','Discount_Normalized',
                'MRP_Zscore','MRP_Log']].describe())

       Rating_Normalized  Discount_Normalized    MRP_Zscore     MRP_Log
count         800.000000           800.000000  8.000000e+02  800.000000
mean            0.691288             0.278750 -4.440892e-18    8.255406
std             0.189845             0.210068  1.000626e+00    1.298377
min             0.000000             0.000000 -5.480385e-01    5.242910
25%             0.575758             0.100000 -4.747905e-01    7.271163
50%             0.696970             0.300000 -3.458446e-01    8.198906
75%             0.818182             0.400000 -6.781716e-02    9.033341
max             1.000000             1.000000  5.589444e+00   11.560226


In [18]:
#Encoding Categorical Variables

# 1. One Hot Encoding — Category
cat_dummies = pd.get_dummies(
    products['Category'], prefix='Cat', drop_first=False
)
products = pd.concat([products, cat_dummies], axis=1)

# 2. One Hot Encoding — Price_Tier (ordinal encoding better)
tier_order = {'Budget':1,'Mid-Range':2,'Premium':3,'Luxury':4,'Unknown':0}
products['Price_Tier_Encoded'] = products['Price_Tier'].map(tier_order)

# 3. Ordinal Encoding — Rating_Band
rating_order = {'Poor':1,'Average':2,'Good':3,'Excellent':4,'Unknown':0}
products['Rating_Band_Encoded'] = products['Rating_Band'].map(rating_order)

# 4. Binary Encoding — Is_Active
products['Is_Active'] = (
    products['Product_Status'] == 'Active'
).astype(int)

# 5. Stock Status Encoding
stock_order = {'Out of Stock':0,'Low Stock':1,'Adequate':2,'High Stock':3,'Unknown':-1}
products['Stock_Status_Encoded'] = products['Stock_Status'].map(stock_order)

print(products[['Price_Tier_Encoded','Rating_Band_Encoded',
                'Is_Active','Stock_Status_Encoded']].head(10))

   Price_Tier_Encoded  Rating_Band_Encoded  Is_Active  Stock_Status_Encoded
0                   3                    3          0                     2
1                   2                    2          0                     3
2                   2                    4          0                     2
3                   4                    3          0                     3
4                   3                    3          1                     3
5                   3                    2          1                     1
6                   3                    2          0                     3
7                   3                    3          0                     3
8                   4                    3          1                     3
9                   3                    3          0                     3


In [19]:
# Final Validation
print("\n" + "="*55)
print("  PRODUCTS — FINAL DATA QUALITY REPORT")
print("="*55)
print(f"  Total rows          : {len(products):,}")
print(f"  Total columns       : {len(products.columns)}")
print(f"  Remaining nulls     : {products.isnull().sum().sum():,}")
print(f"  Duplicates          : {products.duplicated().sum()}")
print(f"  Loss-making products: {products['Is_Loss_Maker'].sum()}")
print(f"  Out of stock        : {(products['Stock_Status']=='Out of Stock').sum()}")
print(f"  High discount prods : {products['High_Discount_Flag'].sum()}")

print(f"\n  CATEGORY DISTRIBUTION:")
print(products['Category'].value_counts())

print(f"\n  PRICE TIER DISTRIBUTION:")
print(products['Price_Tier'].value_counts())

print(f"\n  RATING BAND DISTRIBUTION:")
print(products['Rating_Band'].value_counts())

print(f"\n  GROSS MARGIN BY CATEGORY:")
print(products.groupby('Category')['Gross_Margin_Pct'].describe().round(2))


  PRODUCTS — FINAL DATA QUALITY REPORT
  Total rows          : 800
  Total columns       : 39
  Remaining nulls     : 0
  Duplicates          : 0
  Loss-making products: 87
  Out of stock        : 133
  High discount prods : 147

  CATEGORY DISTRIBUTION:
Category
Electronics       268
Fashion           253
Home & Kitchen    143
Beauty            136
Name: count, dtype: int64

  PRICE TIER DISTRIBUTION:
Price_Tier
Premium      368
Mid-Range    253
Luxury       135
Budget        44
Name: count, dtype: int64

  RATING BAND DISTRIBUTION:
Rating_Band
Average      289
Good         276
Excellent    191
Poor          44
Name: count, dtype: int64

  GROSS MARGIN BY CATEGORY:
                count   mean     std      min    25%    50%    75%    max
Category                                                                 
Beauty          136.0  41.46   20.78  -127.21  36.29  43.75  51.83  86.25
Electronics     268.0  -7.45  121.72  -977.47   3.28  15.04  24.24  92.79
Fashion         253.0  29.82

In [20]:
# Save full cleaned version
products.to_csv('KartZone_Products_Clean.csv', index=False)
print("Saved → KartZone_Products_Clean.csv ✓")

# Save essential columns for SQL load
essential_cols = [
    'Product_ID','Product_Name','Category','Sub_Category','Brand',
    'MRP','Cost_Price','Discount_Pct','Selling_Price',
    'Launch_Date','Rating','Stock_Units','Weight_KG','Product_Status',
    'Profit_Per_Unit','Gross_Margin_Pct','Is_Loss_Maker',
    'Product_Age_Days','Product_Age_Band','Rating_Band',
    'Stock_Status','High_Discount_Flag','Price_Tier','Discount_Amount',
    'Is_Active'
]
products[essential_cols].to_csv('KartZone_Products_Final.csv', index=False)
print("Saved → KartZone_Products_Final.csv ✓")

Saved → KartZone_Products_Clean.csv ✓
Saved → KartZone_Products_Final.csv ✓


In [21]:
print(products['Selling_Price'].describe())

print(products[['MRP', 'Selling_Price', 'Price_Tier']].head(20))

count      800.000000
mean      8069.803162
std      14480.414798
min        142.930000
25%       1246.817500
50%       3296.000000
75%       6970.320000
max      89116.330000
Name: Selling_Price, dtype: float64
          MRP  Selling_Price Price_Tier
0    5493.590        4669.55    Premium
1     909.980         864.48  Mid-Range
2     627.310         564.58  Mid-Range
3   61954.360       55758.92     Luxury
4    2715.150        2579.39    Premium
5    2607.470        2477.10    Premium
6    6407.840        5767.06    Premium
7    7977.265        6780.68    Premium
8   47545.800       40413.93     Luxury
9    2714.640        2578.91    Premium
10   1013.580         912.22  Mid-Range
11   6761.870        5071.40    Premium
12   1226.050        1103.44  Mid-Range
13   8886.680        8442.35    Premium
14   2054.840        2054.84    Premium
15   6978.570        5931.78    Premium
16    488.490         415.22     Budget
17   2408.040        2287.64    Premium
18   2766.350        2628.03

In [22]:
products.columns

Index(['Product_ID', 'Product_Name', 'Category', 'Sub_Category', 'Brand',
       'MRP', 'Cost_Price', 'Discount_Pct', 'Selling_Price', 'Launch_Date',
       'Rating', 'Stock_Units', 'Weight_KG', 'Product_Status',
       'Profit_Per_Unit', 'Gross_Margin_Pct', 'Is_Loss_Maker',
       'Product_Age_Days', 'Product_Age_Band', 'Rating_Band', 'Stock_Status',
       'High_Discount_Flag', 'Price_Tier', 'Discount_Amount',
       'Rating_Normalized', 'Discount_Normalized', 'MRP_Zscore',
       'Profit_Zscore', 'MRP_Log', 'Cost_Log', 'Stock_Log', 'Cat_Beauty',
       'Cat_Electronics', 'Cat_Fashion', 'Cat_Home & Kitchen',
       'Price_Tier_Encoded', 'Rating_Band_Encoded', 'Is_Active',
       'Stock_Status_Encoded'],
      dtype='object')

In [23]:
products.groupby('Category')[

_IncompleteInputError: incomplete input (1147377447.py, line 1)